# Reconstructor: VGGT-Omega

---
---
---

# Environment Context
Force the working directory to the repository root to ensure configuration files and submodules resolve correctly.

In [ ]:
import os
import sys
from pathlib import Path

# Force the working directory to the repository root
if Path.cwd().name == "notebooks":
    os.chdir("..")

# Expose VGGT-Omega modules
vggt_path = Path("external/vggt-omega").resolve()
if str(vggt_path) not in sys.path:
    sys.path.append(str(vggt_path))

# Verify context is now repository root
print(f"Current Working Directory: {Path.cwd()}")

---
---
---

---
---
---

# Step 1: Choose Input-Sequence

## DATASET: TUM RGB-D

In [ ]:
# input_sequence_dir = Path("/storage/group/dataset_mirrors/tum_rgbd_benchmark/rgbd_dataset_freiburg1_room/rgb")
# output_results_dir_base = Path("data/reconstruction/vista_slam_output/tum_rgbd_benchmark/rgbd_dataset_freiburg1_room")

---
---

## DATASET: ScanNet

## Baseline ("EASY" room scenes)

### scene0043_00:

numColorFrames = 1562

In [ ]:
input_sequence_dir = Path("/storage/group/dataset_mirrors/scannet/scans/scene0043_00/color")
output_results_dir_base = Path("data/reconstruction/vggt_omega_output/scannet/scene0043_00")

### scene0022_00:

In [ ]:
# input_sequence_dir = Path("/storage/group/dataset_mirrors/scannet/scans/scene0022_00/color")
# output_results_dir_base = Path("data/reconstruction/vggt_omega_output/scannet/scene0022_00")

---

### Dynamic Path

In [ ]:
# # ==========================================
# # USER INPUT: Define ScanNet Scene ID
# # ==========================================
# scene_id = "scene0049_00"
# # ==========================================

# # Dynamically construct base input and output directories
# dataset_root = Path("/storage/group/dataset_mirrors/scannet/scans")
# output_root = Path("data/reconstruction/vggt_omega_output/scannet")

# input_sequence_dir = dataset_root / scene_id / "color"
# output_results_dir_base = output_root / scene_id

# # Enforce explicit validation
# if not input_sequence_dir.exists():
#     raise FileNotFoundError(f"❌ Error: Input directory does not exist at {input_sequence_dir}")

# print(f"📁 Active Scene ID:        {scene_id}")
# print(f"📥 Input Sequence Path:    {input_sequence_dir}")
# print(f"📤 Output Results Path:    {output_results_dir_base}")

10 room "hard"

17 room "medium"

20 interesting vggt repetitive 

22 big room "medium"

24 room "hard"

34 wc "medium"

40 room "hard"

43 room "easy"

46 room "hard"

47 room "medium"

49 room "hard"



---
---
---

# Step 2: Define config

In [ ]:
# ===============================================================================
# USER INPUT: Define config version (distinct name to avoid override of results!)
config_number = "03"
# ===============================================================================

config_filename = f"custom_3DRoomSearch_v{config_number}.yaml"
abs_output_dir = (output_results_dir_base / f"config_{config_number}").resolve()
abs_output_dir.mkdir(parents=True, exist_ok=True)
abs_config_path = (abs_output_dir / f"{config_filename}").resolve()

# Unified metadata standard across pipelines
meta_path = abs_output_dir / "sequence_mapping.json"

# VGGT-Omega specific inference tracking outputs
output_tensors_file = abs_output_dir / "vggt_predictions.pt"

## Change config here:

In [ ]:
stride = 20
max_frames = 350

## Input-Sequence Parsing

In [ ]:
import os
import json
from pathlib import Path
from PIL import Image

# Enforce strict variable definition
if "input_sequence_dir" not in locals() or "stride" not in locals() or "max_frames" not in locals():
    raise NameError("❌ Error: 'input_sequence_dir', 'stride', or 'max_frames' is not defined.")

# 1. Parse and apply strict numerical sorting
all_images = sorted(
    list(input_sequence_dir.glob("*.jpg")), 
    key=lambda x: int(x.stem)
)

sampled_paths = all_images[::stride][:max_frames]
image_names = [str(p) for p in sampled_paths]
num_images = len(image_names)

# 2. Compute global valid reconstruction bounding box (Full FoV for VGGT)
if num_images > 0:
    with Image.open(sampled_paths[0]) as img:
        W_orig, H_orig = img.size
        
    global_bbox = {
        "x_min": 0,
        "y_min": 0,
        "x_max": W_orig,
        "y_max": H_orig,
        "description": "Full original resolution. VGGT-Omega dynamic patching avoids cropping for standard ScanNet aspect ratios."
    }
else:
    W_orig, H_orig = None, None
    global_bbox = None

# 3. Build Unified Metadata Dictionary
sequence_metadata = {
    "_comment": "VGGT-Omega unified metadata mapping. Keyframe index maps 1:1 to prediction arrays.",
    "original_resolution": [W_orig, H_orig],
    "global_reconstruction_bbox": global_bbox,
    "frames": []
}

for idx, original_path in enumerate(sampled_paths):
    sequence_metadata["frames"].append({
        "keyframe_index": idx,
        "staged_filename": None,
        "original_filename": original_path.name,
        "absolute_path": str(original_path.resolve())
    })

print(f"Sampled {num_images} frames (Stride: {stride}) for inference.")

## (Optional): Visualization of frames that will be passed to the model

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

print(f"Visualizing {num_images} frames...")

max_cols = 10
cols = min(num_images, max_cols)
rows = (num_images + max_cols - 1) // max_cols

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < num_images:
        img = Image.open(image_names[i])
        ax.imshow(img)
        ax.set_title(Path(image_names[i]).name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
---
---

# (Needed only once) Model Initialization

In [ ]:
import torch
from vggt_omega.models import VGGTOmega
from vggt_omega.utils.load_fn import load_and_preprocess_images
from vggt_omega.utils.pose_enc import encoding_to_camera

checkpoint_path = "external/vggt-omega/checkpoints/vggt_omega_1b_512.pt"

# Initialize and load weights
model = VGGTOmega().to("cuda").eval()
model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))

---
---
---

# Step 3: Invoke VGGT-Omega

In [ ]:
# Load and preprocess using the correct, numerically sorted frame paths
images = load_and_preprocess_images(image_names, image_resolution=512).to("cuda")

print(f"Preprocessed tensor shape: {images.shape}")  # Expected: [B, C, H, W]

with torch.inference_mode():
    predictions = model(images)

# Decode camera parameters and append directly to the predictions dictionary
extrinsics, intrinsics = encoding_to_camera(
    predictions["pose_enc"],
    predictions["images"].shape[-2:],
)
predictions["extrinsics"] = extrinsics
predictions["intrinsics"] = intrinsics

print(f"Successfully processed {len(image_names)} frames.")
print(f"Available prediction outputs: {list(predictions.keys())}")

# Store VGGT-Omega outputs

In [ ]:
import torch

# Move predictions to CPU to free VRAM
images_cpu = predictions["images"].cpu()
depth_cpu = predictions["depth"].cpu()
depth_conf_cpu = predictions["depth_conf"].cpu()
ext_cpu = predictions["extrinsics"].cpu()
int_cpu = predictions["intrinsics"].cpu()

# Store model tensor outputs
tensor_dict = {
    "images": images_cpu,
    "depth": depth_cpu,
    "depth_conf": depth_conf_cpu,
    "extrinsics": ext_cpu,
    "intrinsics": int_cpu,
}

torch.save(tensor_dict, output_tensors_file)
print(f"Saved tensor predictions to: {output_tensors_file.name}")

## Config and Metadata Persistence

In [ ]:
import json

# 1. Define and save the inference configuration
config_content = f"""# VGGT-Omega Custom Configuration v{config_number}
stride: {stride}
max_frames: {max_frames}
"""

abs_config_path.write_text(config_content)
print(f"✅ Generated custom configuration: {abs_config_path.name}")

# 2. Write the unified metadata mapping to disk
with open(meta_path, "w") as f:
    json.dump(sequence_metadata, f, indent=4)
print(f"✅ Saved sequence metadata mapping: {meta_path.name}")

---
---
---

# Step 4: Point Cloud Fusion

**Note**: See "Visualization 3: Global Confidence Distribution & Threshold Selection" to choose conf_threshold for fusion

In [ ]:
conf_thresh = 5.9

In [ ]:
import numpy as np
import open3d as o3d
from PIL import Image
import torch

output_pcd_file = abs_output_dir / f"pointcloud_conf_{conf_thresh}.ply"

# 1. Source tensors directly from GPU predictions
# Strip Batch dimension (0) and trailing Channel dimension (-1) from depth
depths = predictions["depth"].squeeze(-1)[0]       # [N, H, W] on CUDA
confs = predictions["depth_conf"][0]               # [N, H, W] on CUDA
exts = predictions["extrinsics"][0]                # [N, 3, 4] on CUDA
ints = predictions["intrinsics"][0]                # [N, 3, 3] on CUDA

N, H, W = depths.shape

# 2. Allocate unprojection grid natively on GPU
v, u = torch.meshgrid(
    torch.arange(H, device="cuda"), 
    torch.arange(W, device="cuda"), 
    indexing="ij"
)
uv = torch.stack([u, v], dim=-1).float()           # [H, W, 2] on CUDA

global_points = []
global_colors = []

for i in range(N):
    depth = depths[i]
    conf = confs[i]
    K = ints[i]
    T_cw = exts[i]

    # Pad to 4x4 and invert to Camera-to-World on GPU
    T_cw_4x4 = torch.eye(4, device="cuda")
    T_cw_4x4[:3, :4] = T_cw
    T_wc = torch.linalg.inv(T_cw_4x4)[:3, :4]

    # Filter invalid/low-confidence geometry
    mask = (conf > conf_thresh) & (depth > 0)
    z = depth[mask]
    uv_valid = uv[mask]

    # Pinhole unprojection to local frame C: Pc = [X, Y, Z, 1]^T
    x = (uv_valid[:, 0] - K[0, 2]) * z / K[0, 0]
    y = (uv_valid[:, 1] - K[1, 2]) * z / K[1, 1]
    pts_local = torch.stack([x, y, z, torch.ones_like(z)], dim=-1) # [M, 4]

    # SE(3) transformation to global frame W
    pts_global = (T_wc @ pts_local.T).T
    
    # 3. Transfer ONLY the highly filtered 3D coordinates to CPU
    global_points.append(pts_global.cpu())

    # Extract aligned RGB values (Read via PIL directly to CPU)
    img = np.array(Image.open(image_names[i]).resize((W, H)))
    colors = torch.from_numpy(img).float() / 255.0
    
    # Apply GPU mask to CPU colors
    global_colors.append(colors[mask.cpu()])

# Fuse and export
all_points = torch.cat(global_points, dim=0).numpy()
all_colors = torch.cat(global_colors, dim=0).numpy()

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(all_points)
pcd.colors = o3d.utility.Vector3dVector(all_colors)

o3d.io.write_point_cloud(str(output_pcd_file), pcd)
print(f"✅ Fused pointcloud saved to: {output_pcd_file.name}")

---
---
---

---
---
---

---
---
---

# Debugging/Observations/Validation

**Note:** Next cells are needed to specify paths of stored outputs for this section:
- One cell from "Step 1: Choose Input-Sequence"
- First cell from "Step 2: Define config"

---

## Visualization 1: VGGT-Omega output shapes

In [ ]:
import torch

if not output_tensors_file.exists():
    raise FileNotFoundError(f"❌ Cannot find saved predictions at {output_tensors_file}. Run inference first.")

# Load serialized tensors into RAM 
predictions = torch.load(output_tensors_file, map_location="cpu")

print(f"Loaded tensors from: {output_tensors_file.name}\n")
print("=" * 50)
print(f"{'Output Key':<20} | {'Tensor Shape'}")
print("-" * 50)

for key, val in predictions.items():
    if isinstance(val, torch.Tensor):
        print(f"{key:<20} | {list(val.shape)}")
    else:
        print(f"{key:<20} | Type: {type(val)}")
        
print("=" * 50)

---

## Observation 1: VGGT-Omega generates outputs for all frames

---

## Observation 2: VGGT-Omega Evaluates the Full Field of View

**Network Input Preprocessing & Field of View:** Unlike strict convolutional or early ViT architectures that enforce a rigid center-crop, VGGT-Omega utilizes a dynamic spatial patching strategy. Assuming the input aspect ratio falls within the network's nominal operating bounds ($[0.5, 2.0]$), the entire image is scaled and padded to align with the 16x16 patch grid without discarding peripheral pixels.

**Implication for Downstream 2D Search:** Any 2D feature matching or object detection executed on the original high-resolution source images maps directly to the generated 3D geometry.

In [ ]:
# ==========================================
# USER INPUT: Set the keyframe index to inspect
target_keyframe_idx = 0
# ==========================================

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 1. Enforce strict variable definition (No fallback)
if "target_keyframe_idx" not in locals() and "target_keyframe_idx" not in globals():
    raise NameError("❌ Error: 'target_keyframe_idx' is not defined. Please set it explicitly before running this cell.")

if not meta_path.exists():
    print(f"❌ Error: Metadata file not found at {meta_path}")
else:
    # 2. Load decoupled metadata from disk
    with open(meta_path, "r") as f:
        sequence_metadata = json.load(f)

    frame_info = next((f for f in sequence_metadata.get("frames", []) if f["keyframe_index"] == target_keyframe_idx), None)

    if not frame_info:
        print(f"❌ Error: Keyframe {target_keyframe_idx} not found in metadata.")
    else:
        original_path = frame_info["absolute_path"]
        
        # Load original high-res image
        with Image.open(original_path) as img:
            W_orig, H_orig = img.size
            img_rgb = np.array(img)

        # 3. Load VGGT-Omega actual output tensor
        predictions = torch.load(output_tensors_file, map_location="cpu")
        
        # Extract image tensor: Strip batch (0) and select keyframe index
        # Shape transition: [B, N, C, H, W] -> [C, H, W]
        vggt_img_tensor = predictions["images"][0, target_keyframe_idx]
        
        # Convert to HWC format for matplotlib (PyTorch uses CHW)
        vggt_img_np = vggt_img_tensor.permute(1, 2, 0).numpy()
        
        # VGGT pads with 1.0. Clip strictly to [0, 1] to prevent matplotlib rendering warnings
        vggt_img_np = np.clip(vggt_img_np, 0, 1)

        # 4. Visualization
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Original Image
        axes[0].imshow(img_rgb)
        axes[0].set_title(f"Source Image ({W_orig}x{H_orig})\nNo Spatial Cropping Applied", fontsize=12)
        axes[0].axis('off')

        # VGGT-Omega Serialized Output
        H_vggt, W_vggt = vggt_img_np.shape[:2]
        axes[1].imshow(vggt_img_np)
        axes[1].set_title(f"Network Output (vggt_predictions.pt)\nResolution: ({W_vggt}x{H_vggt})", fontsize=12)
        axes[1].axis('off')

        plt.suptitle(f"Preprocessing Field of View Validation | keyframe_index = {target_keyframe_idx} | Source: {frame_info['original_filename']}", fontsize=14, y=1.05)
        plt.tight_layout()
        plt.show()

---

# Visualization 2: Per Keyframe VGGT-Omega Predictions

In [ ]:
# ==========================================
# USER INPUT: Set the keyframe index to inspect
target_keyframe_idx = 0
# ==========================================

# ==========================================
# USER INPUT: Adjust confidence threshold
conf_thresh = 5.9
# ==========================================

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1. Enforce strict variable definition
if "target_keyframe_idx" not in locals() and "target_keyframe_idx" not in globals():
    raise NameError("❌ Error: 'target_keyframe_idx' is not defined. Please set it explicitly before running this cell.")

if not output_tensors_file.exists() or not meta_path.exists():
    print("❌ Error: Output tensors or metadata not found. Ensure VGGT-Omega inference completed successfully.")
else:
    # 2. Load decoupled metadata from disk
    with open(meta_path, "r") as f:
        sequence_metadata = json.load(f)

    # Load VGGT predictions
    predictions = torch.load(output_tensors_file, map_location="cpu")
    
    # Extract tensors and strip batch dimension [0]
    images_tensor = predictions["images"][0]
    depths_tensor = predictions["depth"][0]
    confs_tensor = predictions["depth_conf"][0]
    extrinsics_tensor = predictions["extrinsics"][0]
    intrinsics_tensor = predictions["intrinsics"][0]

    N = depths_tensor.shape[0]
    
    if target_keyframe_idx >= N or target_keyframe_idx < 0:
        print(f"❌ Error: target_keyframe_idx {target_keyframe_idx} is out of bounds for sequence length ({N}).")
    else:
        # Extract target frame data and convert to numpy
        # Permute CHW -> HWC and clip for visualization
        rgb_img = np.clip(images_tensor[target_keyframe_idx].permute(1, 2, 0).numpy(), 0, 1)
        
        # Squeeze trailing channel dimension [H, W, 1] -> [H, W]
        raw_depth_map = depths_tensor[target_keyframe_idx].squeeze(-1).numpy()
        conf_map = confs_tensor[target_keyframe_idx].numpy()
        
        K = intrinsics_tensor[target_keyframe_idx].numpy()
        pose = extrinsics_tensor[target_keyframe_idx].numpy() # Shape [3, 4]

        frame_info = next((f for f in sequence_metadata.get("frames", []) if f["keyframe_index"] == target_keyframe_idx), None)
        source_name = frame_info['original_filename'] if frame_info else f"Index {target_keyframe_idx}"
        
        thresholded_depth = np.copy(raw_depth_map)
        thresholded_depth[conf_map < conf_thresh] = np.nan
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # --- ROW 1 ---
        axes[0, 0].imshow(rgb_img)
        axes[0, 0].set_title("Network RGB Input (No Spatial Crop)", fontsize=12)
        axes[0, 0].axis('off')
        
        im_depth = axes[0, 1].imshow(raw_depth_map, cmap='plasma')
        axes[0, 1].set_title("Raw Depth Map", fontsize=12)
        axes[0, 1].axis('off')
        fig.colorbar(im_depth, ax=axes[0, 1], fraction=0.046, pad=0.04, label="Depth (Implicit Scale)")

        # Inject an invisible colorbar on the RGB axis to force spatial alignment
        cbar_dummy = fig.colorbar(im_depth, ax=axes[0, 0], fraction=0.046, pad=0.04)
        cbar_dummy.ax.set_visible(False)
        
        # --- ROW 2 ---
        im_conf = axes[1, 0].imshow(conf_map, cmap='viridis')
        axes[1, 0].set_title("Depth Confidence Map", fontsize=12)
        axes[1, 0].axis('off')
        fig.colorbar(im_conf, ax=axes[1, 0], fraction=0.046, pad=0.04, label="Confidence Score [0, 1]")
        
        cmap_thresh = plt.cm.plasma.copy()
        cmap_thresh.set_bad(color='black')
        
        im_thresh = axes[1, 1].imshow(thresholded_depth, cmap=cmap_thresh)
        axes[1, 1].set_title(f"Thresholded Depth (Conf > {conf_thresh})", fontsize=12)
        axes[1, 1].axis('off')
        fig.colorbar(im_thresh, ax=axes[1, 1], fraction=0.046, pad=0.04, label="Depth (Implicit Scale)")
        
        plt.suptitle(f"VGGT-Omega Outputs | Keyframe: {target_keyframe_idx} | Source: {source_name}", fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()

        # 3. Print extracted geometric parameters
        print(f"\n⚙️ Geometric Parameters for Keyframe: {target_keyframe_idx}")
        print("-" * 50)
        print("📷 Camera Intrinsics (K):")
        print(K)
        print("\n📍 SE(3) Trajectory Pose (Extrinsics 3x4 Full Precision):")
        print(pose)
        print("\n📍 SE(3) Trajectory Pose (Approximation):")
        print(np.round(pose, 2))

---
---
---

## Visualization 3: Global Confidence Distribution & Threshold Selection

Unlike traditional probability maps bounded to $[0, 1]$, VGGT-Omega outputs an unnormalized confidence metric (logit) representing inverse observation variance. Consequently, a static threshold (e.g., $0.5$) is mathematically arbitrary and scene-dependent.

To empirically derive a robust filter for 3D point cloud fusion, we evaluate the global Probability Density Function (PDF) of the confidence sequence. The resulting survival curve maps any given threshold directly to the percentage of total scene geometry it will retain.

**Action:** Analyze the statistical distribution below. Identify a `conf_thresh` value that optimally balances outlier rejection (removing boundary noise) with structural density, and apply it to the final unprojection step.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

if "predictions" not in locals():
    predictions = torch.load(output_tensors_file, map_location="cpu")

# Extract confidence tensor [N, H, W]
confs_tensor = predictions["depth_conf"][0]

# Memory-safe spatial subsampling (sample every 10th pixel across H and W)
# Shape transition: [N, H, W] -> [N, H/10, W/10] -> 1D Array
sampled_confs = confs_tensor[:, ::10, ::10].flatten().numpy()

# Compute survival curve (percentage of pixels retained)
max_conf = float(sampled_confs.max())
min_conf = float(sampled_confs.min())
thresholds = np.linspace(min_conf, max_conf, 100)
survival_rates = [(sampled_confs > t).mean() * 100 for t in thresholds]

# Extract key statistical percentiles
percentiles = [5, 10, 25, 50, 75, 90]
p_values = np.percentile(sampled_confs, percentiles)

# ==========================================
# Visualization
# ==========================================
fig, ax1 = plt.subplots(figsize=(12, 6))

# Primary Axis: Histogram (PDF)
color_hist = 'tab:blue'
ax1.hist(sampled_confs, bins=100, color=color_hist, alpha=0.6, density=True)
ax1.set_xlabel("Unnormalized Confidence Score (Logits)", fontsize=11)
ax1.set_ylabel("Pixel Density", color=color_hist, fontsize=11)
ax1.tick_params(axis='y', labelcolor=color_hist)
ax1.grid(True, alpha=0.3)

# Secondary Axis: Survival Curve (1 - CDF)
ax2 = ax1.twinx()
color_curve = 'tab:red'
ax2.plot(thresholds, survival_rates, color=color_curve, linewidth=2.5)
ax2.set_ylabel("Geometry Retained (%)", color=color_curve, fontsize=11)
ax2.tick_params(axis='y', labelcolor=color_curve)
ax2.set_ylim(0, 105)

plt.title(f"Global Sequence Confidence Distribution & Threshold Survival Rate\n(Subsampled {len(sampled_confs):,} pixels)", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

# ==========================================
# Statistical Summary
# ==========================================
print("⚙️ Global Confidence Percentiles:")
print("-" * 50)
for p, val in zip(percentiles, p_values):
    retention = (sampled_confs > val).mean() * 100
    print(f"[{p:02d}th Percentile] Score: {val:>6.2f}  --> Retains {retention:>6.2f}% of total pixels")
print("-" * 50)
print(f"Absolute Minimum: {min_conf:.2f}")
print(f"Absolute Maximum: {max_conf:.2f}")

---

## Validating VGGT-Omega Predictions

Because VGGT-Omega dynamically resizes the input to conform to a token-balanced $16 \times 16$ patch grid, we cannot directly compare its predicted geometry to the raw ScanNet ground truth. We must forward-project the Ground Truth (GT) parameters through the spatial preprocessing pipeline.

## 1. Camera Intrinsics Validation

Unlike rigid crop methods, VGGT-Omega preserves the entire field of view for nominal aspect ratios. The spatial transformation consists purely of an independent continuous resize along the $X$ and $Y$ axes. By extracting the exact spatial dimensions of the network's input tensor and calculating the respective scaling factors, we derive the "Expected" intrinsics for the rescaled space and compare them against the temporal average of VGGT-Omega's predictions.

In [ ]:
import json
import torch
import numpy as np

# Enforce strict variable definition
if "input_sequence_dir" not in locals() or "abs_output_dir" not in locals():
    raise NameError("❌ Error: 'input_sequence_dir' or 'abs_output_dir' is not defined.")

gt_intrinsic_path = input_sequence_dir.parent / "intrinsic" / "intrinsic_color.txt"

required_files = [gt_intrinsic_path, meta_path, output_tensors_file]

if not all(f.exists() for f in required_files):
    print("❌ Error: Required files for intrinsic validation not found.")
    for f in required_files:
        if not f.exists():
            print(f"  Missing: {f}")
else:
    # 1. Load Raw GT Intrinsics
    gt_K = np.loadtxt(gt_intrinsic_path)
    fx_gt, fy_gt = gt_K[0, 0], gt_K[1, 1]
    cx_gt, cy_gt = gt_K[0, 2], gt_K[1, 2]

    # 2. Load Spatial Metadata
    with open(meta_path, "r") as f:
        meta = json.load(f)
    
    W_orig, H_orig = meta.get("original_resolution", [1296, 968])

    # 3. Load VGGT-Omega Predictions and Extract Target Grid
    predictions = torch.load(output_tensors_file, map_location="cpu")
    
    # Extract tensor spatial dimensions [H, W] to compute exact analytical scales
    H_scaled, W_scaled = predictions["images"].shape[-2:]
    
    # 4. Compute Forward Projection of GT Intrinsics (Pure Resize)
    scale_x = W_scaled / W_orig
    scale_y = H_scaled / H_orig
    
    fx_derived = fx_gt * scale_x
    fy_derived = fy_gt * scale_y
    cx_derived = cx_gt * scale_x
    cy_derived = cy_gt * scale_y

    # 5. Load, Average, and Compute Variance of Predicted Intrinsics
    pred_K_all = predictions["intrinsics"][0].numpy()
    pred_K_avg = np.mean(pred_K_all, axis=0)
    pred_K_std = np.std(pred_K_all, axis=0)
    
    fx_pred, fy_pred = pred_K_avg[0, 0], pred_K_avg[1, 1]
    cx_pred, cy_pred = pred_K_avg[0, 2], pred_K_avg[1, 2]
    
    fx_std, fy_std = pred_K_std[0, 0], pred_K_std[1, 1]
    cx_std, cy_std = pred_K_std[0, 2], pred_K_std[1, 2]

    # 6. Output Comparison
    print(f"📊 Intrinsics Validation: {input_sequence_dir.parent.name}")
    print("=" * 65)
    
    print(f"Original ScanNet Resolution: {W_orig}x{H_orig}")
    print(f"Network Evaluated Resolution: {W_scaled}x{H_scaled}")
    print(f"Raw ScanNet Focal Lengths:   fx = {fx_gt:.2f}, fy = {fy_gt:.2f}")
    print(f"Raw ScanNet Principal Point: cx = {cx_gt:.2f}, cy = {cy_gt:.2f}")
    print("-" * 65)
    
    print("Derived Ground Truth (Mapped to Network Input Grid):")
    print(f"  fx = {fx_derived:>8.3f} | fy = {fy_derived:>8.3f}")
    print(f"  cx = {cx_derived:>8.3f} | cy = {cy_derived:>8.3f}")
    print("-" * 65)
    
    print("VGGT-Omega Average Prediction (± Std Dev):")
    print(f"  fx = {fx_pred:>8.3f} ± {fx_std:<5.3f} | fy = {fy_pred:>8.3f} ± {fy_std:<5.3f}")
    print(f"  cx = {cx_pred:>8.3f} ± {cx_std:<5.3f} | cy = {cy_pred:>8.3f} ± {cy_std:<5.3f}")
    print("-" * 65)
    
    # Compute absolute relative errors against the mean
    err_fx = abs(fx_derived - fx_pred) / fx_derived * 100
    err_fy = abs(fy_derived - fy_pred) / fy_derived * 100
    err_cx = abs(cx_derived - cx_pred) / cx_derived * 100
    err_cy = abs(cy_derived - cy_pred) / cy_derived * 100
    
    print("Prediction Error (Mean vs Derived GT):")
    print(f"  Δfx: {err_fx:.2f}% | Δfy: {err_fy:.2f}%")
    print(f"  Δcx: {err_cx:.2f}% | Δcy: {err_cy:.2f}%")

---

### 2. Depth Map Validation (Visual & Scale Alignment)

To validate the metric depth predictions, we must address two primary challenges: **Spatial Alignment** and **Monocular Scale Ambiguity**.

* **Spatial Alignment:** ScanNet's raw RGB images are captured at a higher resolution (e.g., $1296 \times 968$) than the corresponding depth sensor outputs (e.g., $640 \times 480$). Because VGGT-Omega evaluates the full field of view mapped to a dynamic patch grid (e.g., $592 \times 448$), we must load the raw 16-bit depth image and directly resize it to match the network's evaluated spatial dimensions.
    * > **Critical Detail:** We strictly use **Nearest-Neighbor interpolation** during this resize. Bilinear or bicubic interpolation would average valid depth pixels with invalid ($0$) pixels, creating false depth gradients at object boundaries.
* **Scale Alignment (Median Alignment):** VGGT-Omega predicts depth within an implicit, structurally consistent but uncalibrated metric scale. To evaluate the actual *structural* quality against ScanNet ground truth, we apply **Median Alignment**. By calculating a global scalar ($s_{align}$) derived from the median ratio of valid ground truth pixels to predicted pixels, we linearly shift the predicted depth map into the ground truth metric space for a 1:1 comparison.

In [ ]:
# ==========================================
# USER INPUT: Set the keyframe index to inspect
target_keyframe_idx = 0
# ==========================================

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# 1. Enforce strict variable definition
if "target_keyframe_idx" not in locals() and "target_keyframe_idx" not in globals():
    raise NameError("❌ Error: 'target_keyframe_idx' is not defined. Please set it explicitly.")
if "input_sequence_dir" not in locals() or "abs_output_dir" not in locals():
    raise NameError("❌ Error: Base directories are not defined.")

meta_path = abs_output_dir / "sequence_mapping.json"
depth_dir = input_sequence_dir.parent / "depth"

required_files = [meta_path, output_tensors_file]

if not all(f.exists() for f in required_files) or not depth_dir.exists():
    print("❌ Error: Output tensors, metadata, or ScanNet /depth/ directory not found.")
else:
    with open(meta_path, "r") as f:
        meta = json.load(f)
    
    frame_info = next((f for f in meta.get("frames", []) if f["keyframe_index"] == target_keyframe_idx), None)
    
    if not frame_info:
        print(f"❌ Error: Keyframe {target_keyframe_idx} not found in metadata.")
    else:
        original_rgb_name = frame_info["original_filename"]
        frame_stem = Path(original_rgb_name).stem
        gt_depth_path = depth_dir / f"{frame_stem}.png"
        
        if not gt_depth_path.exists():
            print(f"❌ Error: Ground truth depth map not found at {gt_depth_path}")
        else:
            # 2. Load VGGT-Omega Predictions & Target Dimensions
            predictions = torch.load(output_tensors_file, map_location="cpu")
            
            # Extract target spatial resolution [H, W] directly from network input
            H_scaled, W_scaled = predictions["images"].shape[-2:]
            
            # 3. Process Ground Truth Depth
            with Image.open(gt_depth_path) as depth_img:
                # Resize directly to network grid using Nearest Neighbor
                resample_method = getattr(Image, 'Resampling', Image).NEAREST
                depth_resized = depth_img.resize((W_scaled, H_scaled), resample_method)
                
                # ScanNet depth is stored in millimeters. Convert to meters.
                gt_depth_np = np.array(depth_resized).astype(np.float32) / 1000.0
                gt_depth_np[gt_depth_np == 0.0] = np.nan
            
            # 4. Extract Predicted Depth
            raw_pred_depth = predictions["depth"][0, target_keyframe_idx].squeeze(-1).numpy()
            
            # 5. MEDIAN ALIGNMENT LOGIC
            # Find pixels valid in both maps to compute the shift
            valid_mask = (~np.isnan(gt_depth_np)) & (~np.isnan(raw_pred_depth)) & (raw_pred_depth > 0)
            
            if np.sum(valid_mask) > 0:
                s_align = np.nanmedian(gt_depth_np[valid_mask] / raw_pred_depth[valid_mask])
                aligned_pred_depth = raw_pred_depth * s_align
            else:
                print("⚠️ Warning: No valid overlapping pixels found for median alignment.")
                s_align = 1.0
                aligned_pred_depth = raw_pred_depth
            
            # Extract corresponding RGB for visualization
            rgb_img = np.clip(predictions["images"][0, target_keyframe_idx].permute(1, 2, 0).numpy(), 0, 1)

            # 6. Visualization Setup
            vmin = min(np.nanmin(aligned_pred_depth), np.nanmin(gt_depth_np))
            vmax = max(np.nanmax(aligned_pred_depth), np.nanmax(gt_depth_np))
            
            cmap = plt.cm.plasma.copy()
            cmap.set_bad(color='black')
            
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            
            axes[0].imshow(rgb_img)
            axes[0].set_title(f"Network Evaluated RGB ({W_scaled}x{H_scaled})", fontsize=12)
            axes[0].axis('off')
            
            im_pred = axes[1].imshow(aligned_pred_depth, cmap=cmap, vmin=vmin, vmax=vmax)
            axes[1].set_title(f"Median-Aligned Predicted Depth (m)\n(Derived Scale: {s_align:.3f})", fontsize=12)
            axes[1].axis('off')
            fig.colorbar(im_pred, ax=axes[1], fraction=0.046, pad=0.04)
            
            im_gt = axes[2].imshow(gt_depth_np, cmap=cmap, vmin=vmin, vmax=vmax)
            axes[2].set_title(f"GT Depth Nearest-Neighbor ({W_scaled}x{H_scaled})", fontsize=12)
            axes[2].axis('off')
            fig.colorbar(im_gt, ax=axes[2], fraction=0.046, pad=0.04)
            
            plt.suptitle(f"Depth Map Validation | Keyframe: {target_keyframe_idx} | Source: {original_rgb_name}", fontsize=14, y=1.05)
            plt.tight_layout()
            plt.show()

### 3. Quantitative Depth Evaluation (Sequence-Level)

This section computes the Absolute Relative Error (AbsRel) and Root Mean Square Error (RMSE) across the entire sequence. It implements a single, global Median Alignment scalar to decouple structural prediction error from monocular trajectory scale drift. 

To evaluate the impact of preprocessing thresholds on the final 3D reconstruction, metrics are computed across two ablation tiers:
1. **Tier 1 (Raw):** All valid pixels.
2. **Tier 2 (Confidence):** Valid pixels where $Conf \ge Threshold$.

In [ ]:
# ==========================================
# USER INPUT: Evaluation Thresholds
# ==========================================
conf_thres = 8.5       # Minimum empirical confidence score (VGGT logit space)
# ==========================================

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# 1. Enforce strict variable definition
required_vars = ["input_sequence_dir", "abs_output_dir", "conf_thres"]
for var in required_vars:
    if var not in locals() and var not in globals():
        raise NameError(f"❌ Error: '{var}' is not defined.")

meta_path = abs_output_dir / "sequence_mapping.json"
depth_dir = input_sequence_dir.parent / "depth"

required_files = [meta_path, output_tensors_file]

if not all(f.exists() for f in required_files) or not depth_dir.exists():
    print("❌ Error: Required output tensors, metadata, or ScanNet /depth/ directory not found.")
else:
    # 2. Load Data
    with open(meta_path, "r") as f:
        meta = json.load(f)
    frames = meta.get("frames", [])
    
    predictions = torch.load(output_tensors_file, map_location="cpu")
    
    # Extract resolution directly from outputs
    H_scaled, W_scaled = predictions["images"].shape[-2:]
    
    depths_tensor = predictions["depth"][0].squeeze(-1).numpy()
    confs_tensor = predictions["depth_conf"][0].numpy()

    # 3. Pass 1: Compute Global Scale (Median Alignment)
    valid_ratios = []
    valid_frames = []
    
    for frame in frames:
        idx = frame.get("keyframe_index")
        if idx is None or idx >= depths_tensor.shape[0]:
            continue
            
        original_rgb_name = frame["original_filename"]
        gt_depth_path = depth_dir / f"{Path(original_rgb_name).stem}.png"
        
        if not gt_depth_path.exists():
            continue
            
        with Image.open(gt_depth_path) as depth_img:
            resample_method = getattr(Image, 'Resampling', Image).NEAREST
            depth_resized = depth_img.resize((W_scaled, H_scaled), resample_method)
            gt_np = np.array(depth_resized).astype(np.float32) / 1000.0
            gt_np[gt_np == 0.0] = np.nan
            
        pred_np = depths_tensor[idx]
        
        mask = (~np.isnan(gt_np)) & (~np.isnan(pred_np)) & (pred_np > 0)
        if np.any(mask):
            valid_ratios.append(gt_np[mask] / pred_np[mask])
            valid_frames.append((idx, gt_np, pred_np, confs_tensor[idx]))

    if not valid_ratios:
        raise ValueError("❌ Error: No valid overlapping pixels found in the entire sequence to compute global scale.")
        
    global_s_align = np.nanmedian(np.concatenate(valid_ratios))
    
    # 4. Pass 2: Compute Metrics per Frame
    metrics = {"idx": [], "t1_absrel": [], "t1_rmse": [], "t2_absrel": [], "t2_rmse": []}
    
    for idx, gt_np, raw_pred_np, conf_np in valid_frames:
        aligned_pred = raw_pred_np * global_s_align
        
        # Define Masks
        mask_t1 = (~np.isnan(gt_np)) & (~np.isnan(aligned_pred))
        mask_t2 = mask_t1 & (conf_np >= conf_thres)
        
        def calc_metrics(m):
            if not np.any(m): return np.nan, np.nan
            absrel = np.mean(np.abs(gt_np[m] - aligned_pred[m]) / gt_np[m])
            rmse = np.sqrt(np.mean((gt_np[m] - aligned_pred[m])**2))
            return absrel, rmse

        a1, r1 = calc_metrics(mask_t1)
        a2, r2 = calc_metrics(mask_t2)
        
        metrics["idx"].append(idx)
        metrics["t1_absrel"].append(a1)
        metrics["t1_rmse"].append(r1)
        metrics["t2_absrel"].append(a2)
        metrics["t2_rmse"].append(r2)

    # 5. Output Summary Table
    print(f"📊 Sequence Quantitative Evaluation: {input_sequence_dir.parent.name}")
    print(f"Global Alignment Scale (s_align): {global_s_align:.4f}")
    print("=" * 65)
    print(f"{'Evaluation Tier':<25} | {'Mean AbsRel':<15} | {'Mean RMSE (m)':<15}")
    print("-" * 65)
    
    t1_a, t1_r = np.nanmean(metrics['t1_absrel']), np.nanmean(metrics['t1_rmse'])
    t2_a, t2_r = np.nanmean(metrics['t2_absrel']), np.nanmean(metrics['t2_rmse'])
    
    print(f"{'Tier 1 (All Valid)':<25} | {t1_a:<15.4f} | {t1_r:<15.4f}")
    print(f"{'Tier 2 (Conf Filtered)':<25} | {t2_a:<15.4f} | {t2_r:<15.4f}")
    print("=" * 65)

    # 6. Visualization
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    ax1.plot(metrics["idx"], metrics["t1_absrel"], label="Tier 1 (Raw)", color="red", alpha=0.4, linestyle=":")
    ax1.plot(metrics["idx"], metrics["t2_absrel"], label="Tier 2 (Conf Filtered)", color="blue", linewidth=2)
    ax1.set_ylabel("Absolute Relative Error", fontsize=12)
    ax1.set_title("AbsRel vs. Keyframe Index", fontsize=12)
    ax1.grid(True, linestyle=":", alpha=0.7)
    ax1.legend()

    ax2.plot(metrics["idx"], metrics["t1_rmse"], label="Tier 1 (Raw)", color="red", alpha=0.4, linestyle=":")
    ax2.plot(metrics["idx"], metrics["t2_rmse"], label="Tier 2 (Conf Filtered)", color="green", linewidth=2)
    ax2.set_xlabel("Keyframe Index", fontsize=12)
    ax2.set_ylabel("RMSE (meters)", fontsize=12)
    ax2.set_title("RMSE vs. Keyframe Index", fontsize=12)
    ax2.grid(True, linestyle=":", alpha=0.7)
    ax2.legend()
    
    plt.suptitle(f"VGGT-Omega Depth Error Propagation over Time", fontsize=14)
    plt.tight_layout()
    plt.show()

### 4. Trajectory and Odometry Evaluation

Because VGGT-Omega predicts continuous 3D scene geometry from uncalibrated image streams, its estimated trajectory possesses an arbitrary metric scale and coordinate system ($7$-DoF: $3$ translation, $3$ rotation, $1$ scale) relative to the absolute ScanNet ground truth.

To evaluate its geometric drift and spatial accuracy against standard monocular SLAM benchmarks, we implement the following alignment pipeline:

1. **Sim(3) Trajectory Alignment (Umeyama's Algorithm):** We compute the closed-form optimal rotation ($R \in SO(3)$), translation ($t \in \mathbb{R}^3$), and scale ($s \in \mathbb{R}^+$) that maps the predicted extrinsic translation vectors to the ground truth translation vectors in a least-squares sense.
2. **Absolute Trajectory Error (ATE):** Once globally aligned via Sim(3), we measure spatial drift by computing the Root Mean Square Error (RMSE) between the predicted and ground truth translation paths.
3. **Relative Pose Error (RPE):** To evaluate local odometry consistency (frame-to-frame tracking accuracy), we compute the rotational error ($\Delta \theta$) and scale-corrected translational error ($\Delta t$) over consecutive keyframe transitions.

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Enforce strict variable definition
required_vars = ["input_sequence_dir", "abs_output_dir"]
for var in required_vars:
    if var not in locals() and var not in globals():
        raise NameError(f"❌ Error: '{var}' is not defined.")

meta_path = abs_output_dir / "sequence_mapping.json"
pose_dir = input_sequence_dir.parent / "pose"

if not all(f.exists() for f in [meta_path, output_tensors_file]) or not pose_dir.exists():
    print("❌ Error: Trajectory tensors, metadata, or ScanNet /pose/ directory not found.")
else:
    with open(meta_path, "r") as f:
        meta = json.load(f)
    frames = meta.get("frames", [])
    
    # Load VGGT predictions and extract extrinsics [N, 3, 4]
    predictions = torch.load(output_tensors_file, map_location="cpu")
    extrinsics_tensor = predictions["extrinsics"][0].numpy()
    
    gt_poses = []
    pred_poses = []
    valid_indices = []
    
    # Define OpenCV to OpenGL coordinate system conversion matrix
    cv_to_gl = np.diag([1.0, -1.0, -1.0, 1.0])

    # 1. Temporal Association and Filtering
    for frame in frames:
        idx = frame.get("keyframe_index")
        if idx is None or idx >= extrinsics_tensor.shape[0]:
            continue
            
        frame_stem = Path(frame["original_filename"]).stem
        pose_file = pose_dir / f"{frame_stem}.txt"
        
        if not pose_file.exists():
            continue
            
        gt_pose = np.loadtxt(pose_file)
        if np.any(np.isinf(gt_pose)) or np.any(np.isnan(gt_pose)):
            continue
            
        gt_poses.append(gt_pose)
        
        # Load 3x4 World-to-Camera extrinsic
        ext_3x4 = extrinsics_tensor[idx]
        ext_w2c = np.eye(4)
        ext_w2c[:3, :] = ext_3x4
        
        # Invert to Camera-to-World and transform from OpenCV (+Y Down, +Z Forward) 
        # to ScanNet OpenGL (+Y Up, -Z Forward) convention
        ext_c2w = np.linalg.inv(ext_w2c) @ cv_to_gl
        
        pred_poses.append(ext_c2w)
        valid_indices.append(idx)
        
    if len(gt_poses) < 2:
        raise ValueError("❌ Error: Insufficient valid ground truth poses to perform alignment.")

    gt_poses = np.array(gt_poses)
    pred_poses = np.array(pred_poses)
    
    # Extract translation vectors (3 x N)
    t_gt = gt_poses[:, :3, 3].T
    t_pred = pred_poses[:, :3, 3].T
    
    # 2. Sim(3) Alignment via Umeyama's Algorithm
    mu_gt = np.mean(t_gt, axis=1, keepdims=True)
    mu_pred = np.mean(t_pred, axis=1, keepdims=True)
    
    t_gt_centered = t_gt - mu_gt
    t_pred_centered = t_pred - mu_pred
    
    var_pred = np.mean(np.sum(t_pred_centered**2, axis=0))
    
    H = (t_gt_centered @ t_pred_centered.T) / t_gt.shape[1]
    U, D, Vt = np.linalg.svd(H)
    
    d = np.sign(np.linalg.det(U) * np.linalg.det(Vt))
    S = np.diag([1, 1, d])
    
    R_align = U @ S @ Vt
    c_align = (1.0 / var_pred) * np.trace(np.diag(D) @ S)
    t_align = mu_gt - c_align * (R_align @ mu_pred)
    
    # Apply Sim(3) transformation to predicted translations
    t_pred_aligned = c_align * (R_align @ t_pred) + t_align
    
    # 3. Compute ATE (Absolute Trajectory Error)
    ate_errors = np.linalg.norm(t_gt - t_pred_aligned, axis=0)
    ate_rmse = np.sqrt(np.mean(ate_errors**2))
    ate_median = np.median(ate_errors)
    
    # 4. Compute RPE (Relative Pose Error)
    rpe_delta = 1  # Frame-to-frame step
    rpe_trans_errors = []
    rpe_rot_errors = []
    
    for i in range(len(gt_poses) - rpe_delta):
        j = i + rpe_delta
        
        # Ground Truth Relative Pose
        T_gt_i_inv = np.linalg.inv(gt_poses[i])
        T_gt_rel = T_gt_i_inv @ gt_poses[j]
        t_gt_rel = T_gt_rel[:3, 3]
        R_gt_rel = T_gt_rel[:3, :3]
        
        # Predicted Relative Pose
        T_pred_i_inv = np.linalg.inv(pred_poses[i])
        T_pred_rel = T_pred_i_inv @ pred_poses[j]
        t_pred_rel = c_align * T_pred_rel[:3, 3]
        R_pred_rel = T_pred_rel[:3, :3]
        
        # Translational Error (L2 norm)
        e_trans = np.linalg.norm(t_gt_rel - t_pred_rel)
        rpe_trans_errors.append(e_trans)
        
        # Rotational Error (Axis-Angle distance)
        R_err = R_gt_rel.T @ R_pred_rel
        trace_val = np.clip((np.trace(R_err) - 1.0) / 2.0, -1.0, 1.0)
        e_rot = np.arccos(trace_val) * (180.0 / np.pi)  # Convert to degrees
        rpe_rot_errors.append(e_rot)

    rpe_trans_rmse = np.sqrt(np.mean(np.array(rpe_trans_errors)**2))
    rpe_rot_rmse = np.sqrt(np.mean(np.array(rpe_rot_errors)**2))

    # 5. Output Evaluation Table
    print(f"📊 Trajectory Evaluation: {input_sequence_dir.parent.name}")
    print(f"Computed Sim(3) Scale (c): {c_align:.4f}")
    print("=" * 60)
    print(f"{'Metric':<35} | {'Value'}")
    print("-" * 60)
    print(f"{'Absolute Trajectory Error (RMSE)':<35} | {ate_rmse:.4f} m")
    print(f"{'Absolute Trajectory Error (Median)':<35} | {ate_median:.4f} m")
    print(f"{'Relative Pose Error - Trans (RMSE)':<35} | {rpe_trans_rmse:.4f} m / keyframe")
    print(f"{'Relative Pose Error - Rot (RMSE)':<35} | {rpe_rot_rmse:.4f}° / keyframe")
    print("=" * 60)

    # 6. Visualization (2x2 Dashboard)
    fig = plt.figure(figsize=(16, 12))
    
    # Subplot 1: Top View (X-Y)
    ax1 = fig.add_subplot(221)
    ax1.plot(t_gt[0, :], t_gt[1, :], label="Ground Truth", color="black", linewidth=2)
    ax1.plot(t_pred_aligned[0, :], t_pred_aligned[1, :], 
             label="VGGT-Omega (Aligned)", color="blue", alpha=0.8, linestyle="--")
    ax1.set_title("Top View (X-Y Plane)", fontsize=12)
    ax1.set_xlabel("X (m)")
    ax1.set_ylabel("Y (m)")
    ax1.grid(True, linestyle=":", alpha=0.7)
    ax1.legend()

    # Subplot 2: Side View (X-Z)
    ax2 = fig.add_subplot(222)
    ax2.plot(t_gt[0, :], t_gt[2, :], label="Ground Truth", color="black", linewidth=2)
    ax2.plot(t_pred_aligned[0, :], t_pred_aligned[2, :], 
             label="VGGT-Omega (Aligned)", color="blue", alpha=0.8, linestyle="--")
    ax2.set_title("Side View (X-Z Plane)", fontsize=12)
    ax2.set_xlabel("X (m)")
    ax2.set_ylabel("Elevation Z (m)")
    ax2.grid(True, linestyle=":", alpha=0.7)
    ax2.legend()

    # Subplot 3: 3D Trajectory Map
    ax3 = fig.add_subplot(223, projection='3d')
    ax3.plot(t_gt[0, :], t_gt[1, :], t_gt[2, :], label="Ground Truth", color="black", linewidth=2)
    ax3.plot(t_pred_aligned[0, :], t_pred_aligned[1, :], t_pred_aligned[2, :], 
             label="VGGT-Omega (Aligned)", color="blue", alpha=0.8, linestyle="--")
    ax3.set_title("Global 3D Trajectory", fontsize=12)
    ax3.set_xlabel("X (m)")
    ax3.set_ylabel("Y (m)")
    ax3.set_zlabel("Z (m)")
    
    # Subplot 4: ATE Distribution over Time
    ax4 = fig.add_subplot(224)
    ax4.plot(valid_indices, ate_errors, color="red", linewidth=1.5)
    ax4.fill_between(valid_indices, 0, ate_errors, color="red", alpha=0.1)
    ax4.set_title("Absolute Trajectory Error per Keyframe", fontsize=12)
    ax4.set_xlabel("Keyframe Index")
    ax4.set_ylabel("ATE (meters)")
    ax4.grid(True, linestyle=":", alpha=0.7)
    
    plt.tight_layout()
    plt.show()